# Chapitre 3 · Comment une machine « apprend » (solutions des exercices)

Ce notebook contient **uniquement les réponses aux quatre exercices** du notebook
du chapitre. Le code de la leçon, lui, vit dans le notebook du chapitre et dans
le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte
« IA débranchée » vaut aussi pour les corrigés.

## Mise en place (reprise de la leçon)

Le minimum repris de la leçon pour que les validations tournent ici de façon
autonome : les données de Sètondji, les deux fonctions d'erreur, et la chaîne de
la section 5.

In [ ]:
distances = [2.0, 5.0, 8.0, 12.0]            # km parcourus
prix      = [400.0, 700.0, 1250.0, 1650.0]   # FCFA payes par le client
durees    = [15.0, 12.0, 30.0, 28.0]         # minutes


def erreur(w):
    """Erreur moyenne du modele  prix_predit = w * distance."""
    return sum((w * d - p) ** 2 for d, p in zip(distances, prix)) / len(distances)


def erreur2(w1, w2):
    """Erreur moyenne du modele  prix_predit = w1 * distance + w2 * duree."""
    return sum((w1 * d + w2 * t - p) ** 2
               for d, t, p in zip(distances, durees, prix)) / len(distances)


# La chaine de la section 5 : un seul trajet (8 km, 1 250 FCFA), au point w0 = 100.
d, p_reel = 8.0, 1250.0
carre = lambda e: e ** 2


def prix_pred(w):
    return w * d                          # maillon 1 : w -> prix predit


def erreur_trajet(w):
    return carre(prix_pred(w) - p_reel)   # la chaine entiere : w -> erreur


w0 = 100.0
ecart0 = prix_pred(w0) - p_reel           # l'ecart au point ou l'on se trouve

### Exercice 1 · Le capteur de pente — niveau ●

Trois lignes qui portent tout le chapitre : pousse `x` des deux côtés du point
(`x + h` et `x - h`), et fais le rapport sur la largeur totale du pas.

In [ ]:
def pente(f, x, h=1e-5):
    """De combien f(x) bouge-t-elle quand on pousse x d'un cheveu ?"""
    return (f(x + h) - f(x - h)) / (2 * h)

In [ ]:
# Validation : le capteur de pente.
p = pente(lambda x: x ** 2, 3.0)
assert isinstance(p, float), "le capteur doit renvoyer un nombre"
assert abs(p - 6.0) < 1e-4, f"pente attendue 6.0 en x=3 pour x**2, obtenue {p}"
assert pente(erreur, 100.0) < 0, "en w=100 la pente doit etre negative (il faut monter)"
assert pente(erreur, 200.0) > 0, "en w=200 la pente doit etre positive (il faut descendre)"
print("Capteur OK : pente(x**2, en 3) =", round(p, 6))

### Exercice 2 · Un pas de descente — niveau ●●

Le pas de descente : mesurer la pente de `f` en `w` avec le capteur, puis avancer
à l'opposé, avec un pas de taille `lr`.

In [ ]:
def descendre(f, w, lr=0.005, n_pas=20):
    """Repete des pas de descente et renvoie le w final."""
    for etape in range(n_pas):
        if etape % 4 == 0:
            print(f"etape {etape:2d} | w = {w:7.2f} | erreur = {f(w):>10,.0f}".replace(",", " "))
        w = w - lr * pente(f, w)
    print(f"final    | w = {w:7.2f} | erreur = {f(w):>10,.0f}".replace(",", " "))
    return w


w_final = descendre(erreur, w=50.0)

In [ ]:
# Validation : la descente doit converger vers w* (environ 143.9 FCFA/km).
assert isinstance(w_final, float), "descendre doit renvoyer le w final"
assert abs(w_final - 143.9) < 0.5, f"w final attendu proche de 143.9, obtenu {w_final:.2f}"
assert erreur(w_final) < 7200, "l'erreur finale doit etre au plancher (environ 7 157)"
print(f"Descente OK : w = {w_final:.2f} FCFA/km, erreur = {erreur(w_final):,.0f}".replace(",", " "))

### Exercice 3 · Le gradient numérique — niveau ●●

Deux boutons, donc deux capteurs : on pousse `w1` des deux côtés en gelant `w2`,
puis l'inverse.

In [ ]:
def gradient_numerique(f, w1, w2, h=1e-5):
    """Les deux pentes de f au point (w1, w2) : une par bouton."""
    p1 = (f(w1 + h, w2) - f(w1 - h, w2)) / (2 * h)   # on pousse w1, w2 est gele
    p2 = (f(w1, w2 + h) - f(w1, w2 - h)) / (2 * h)   # on pousse w2, w1 est gele
    return p1, p2

In [ ]:
# Validation : sur g(w1, w2) = w1**2 + 3*w1*w2, les pentes exactes
# au point (2, 1) valent 2*w1 + 3*w2 = 7 (selon w1) et 3*w1 = 6 (selon w2).
g = lambda w1, w2: w1 ** 2 + 3 * w1 * w2
p1, p2 = gradient_numerique(g, 2.0, 1.0)
assert abs(p1 - 7.0) < 1e-4, f"pente selon w1 attendue 7.0, obtenue {p1}"
assert abs(p2 - 6.0) < 1e-4, f"pente selon w2 attendue 6.0, obtenue {p2}"
print(f"Gradient OK : ({p1:.4f}, {p2:.4f})")

# Et sur l'erreur de Setondji, au point (143.88, 0) :
p1, p2 = gradient_numerique(erreur2, 143.88, 0.0)
print(f"gradient de erreur2 en (143.88, 0) : ({p1:.1f}, {p2:.1f})")
print("Le bouton w1 est presque a plat ; c'est le bouton w2 (la duree) qui compte.")

### Exercice 4 · La règle de la chaîne — niveau ●●●

On mesure la pente de chaque maillon au point où lui se trouve (`prix_pred` en
`w0`, `carre` en `ecart0`), on multiplie, et on compare à la pente de la chaîne
entière mesurée au capteur.

In [ ]:
globale = pente(erreur_trajet, w0)       # la pente de la chaine entiere, au capteur
locale_1 = pente(prix_pred, w0)          # maillon 1 : w -> prix predit (pente = 8, la distance)
locale_2 = pente(carre, ecart0)          # maillon 2 : ecart -> ecart**2 (pente = 2 * ecart)
produit = locale_1 * locale_2
print(f"pente globale      : {globale:.1f}")
print(f"pente du maillon 1 : {locale_1:.4f}")
print(f"pente du maillon 2 : {locale_2:.4f}")
print(f"produit des deux   : {produit:.1f}")

In [ ]:
# Validation : produit des pentes locales = pente globale.
assert abs(locale_1 - 8.0) < 1e-3, f"maillon 1 : pente attendue 8.0, obtenue {locale_1}"
assert abs(locale_2 - 2 * ecart0) < 1e-2, f"maillon 2 : pente attendue {2 * ecart0}, obtenue {locale_2}"
assert abs(produit - globale) < 0.1, (
    f"le produit des pentes locales ({produit:.2f}) doit egaler la pente globale ({globale:.2f})"
)
print("Regle de la chaine OK : le produit des pentes locales retombe sur la pente globale.")